# Aligned stability sweep versus the momentum gap $1-\rho$

This experiment complements the earlier batch-size stability scan. It fixes the batch size and scans the momentum damping $g=1-\rho$ for **SGD, Polyak, and Nesterov**. For every $(algorithm,g,\eta)$ and random seed, training is declared stable when the final moving-average population excess loss remains finite and below the configured threshold.

The expected boundaries are

$$\eta_{\rm SGD}^{\rm crit}\asymp 1,\qquad \eta_{\rm Polyak}^{\rm crit}\asymp\min\{1,B(1-\rho)\},\qquad \eta_{\rm Nesterov}^{\rm crit}\asymp\min\{1,B^\beta(1-\rho)\}.$$

Thus both momentum curves should be linear in $1-\rho$ before reaching their deterministic ceilings, with Nesterov shifted upward by its stronger $B^\beta$ dependence.

In [ ]:
from pathlib import Path
import json, math, platform, time

import matplotlib.pyplot as plt
import numpy as np
import torch

if torch.cuda.is_available():
    DEVICE, DTYPE = torch.device('cuda'), torch.float64
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = torch.device('mps'), torch.float32
else:
    DEVICE, DTYPE = torch.device('cpu'), torch.float64

OUTPUT_DIR = Path('stability/results/generated/aligned/momentum_gap')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'device={DEVICE}, dtype={DTYPE}, torch={torch.__version__}, numpy={np.__version__}')

## Experiment setting

This experiment uses the same $\beta=1.5$, $s=0.3$, no-label-noise problem, dimension, learning-rate grid, run length, threshold, and seeds as `batch_size.ipynb`. Its fixed $B=16$ lies in the batch-size grid, and the companion experiment's fixed $1-\rho=2^{-8}$ lies in this momentum-gap grid. The grid excludes $K(1-\rho)\ll1$, where a finite run cannot measure the asymptotic stability boundary.

In [ ]:
PAPER_RUN = True
BETA, S, SIGMA = 1.5, 0.3, 0.0
BATCH_SIZE = 16
if PAPER_RUN:
    DIM = 4096
    ONE_MINUS_RHO_GRID = 2.0 ** np.arange(-8, -3, 0.5, dtype=float)
    ETA_GRID = np.geomspace(2.0 ** -10, 8.0, 50)
    STEPS, WINDOW, N_SEEDS, CHUNK = 2000, 20, 5, 2048
else:
    DIM = 256
    ONE_MINUS_RHO_GRID = 2.0 ** np.arange(-10, 0, 2, dtype=float)
    ETA_GRID = np.geomspace(2.0 ** -10, 4.0, 20)
    STEPS, WINDOW, N_SEEDS, CHUNK = 100, 20, 1, 256
STABLE_THRESHOLD, DIVERGENCE_LOSS = 1.0, 1e12
SEED = 20260822
SEED_SCHEME = 'aligned-v2: algorithm, seed, log2(B), -2log2(1-rho), eta-index'
ALGORITHMS = ('sgd', 'polyak', 'nesterov')
DISPLAY = {'sgd': 'SGD', 'polyak': 'Polyak', 'nesterov': 'Nesterov'}

def aligned_trial_seed(algorithm_index, seed_index, batch_size, one_minus_rho, eta_index):
    """Shared seed map; identical settings give identical samples in both notebooks."""
    batch_code = int(round(np.log2(float(batch_size))))
    gap_half_exponent_code = int(round(-2.0 * np.log2(float(one_minus_rho))))
    return int(
        SEED
        + algorithm_index * 100_000_000
        + seed_index * 10_000_000
        + batch_code * 100_000
        + gap_half_exponent_code * 100
        + eta_index
    )

config = {
    'preset': 'paper' if PAPER_RUN else 'quick', 'beta': BETA, 's': S, 'sigma': SIGMA,
    'dim': DIM, 'batch_size': BATCH_SIZE,
    'one_minus_rho_grid': ONE_MINUS_RHO_GRID.tolist(), 'eta_grid': ETA_GRID.tolist(),
    'steps': STEPS, 'window': WINDOW, 'stable_threshold': STABLE_THRESHOLD,
    'divergence_loss': DIVERGENCE_LOSS, 'n_seeds': N_SEEDS, 'seed': SEED,
    'seed_scheme': SEED_SCHEME,
    'device': str(DEVICE), 'dtype': str(DTYPE), 'python': platform.python_version(),
    'torch': torch.__version__, 'numpy': np.__version__,
}
print(json.dumps(config, indent=2))
print(f'minimum resolved damping time: K(1-rho)={STEPS * ONE_MINUS_RHO_GRID.min():.3f}')

In [ ]:
j = torch.arange(1, DIM + 1, device=DEVICE, dtype=DTYPE)
lam = j.pow(-BETA)
theta_star = j.pow(-0.5 * (1.0 + (S - 1.0) * BETA))
sqrt_lam = lam.sqrt()
initial_loss = float((0.5 * torch.sum(lam * theta_star.square())).item())
print(f'lambda range=[{lam[-1].item():.3e}, {lam[0].item():.3e}], initial loss={initial_loss:.6f}')

@torch.no_grad()
def exact_gradient(delta, batch_size, generator):
    grad = torch.zeros_like(delta)
    remaining = int(batch_size)
    while remaining:
        n = min(CHUNK, remaining)
        x = torch.randn((n, DIM), device=DEVICE, dtype=DTYPE, generator=generator)
        x.mul_(sqrt_lam)
        residual = x.mv(delta)
        if SIGMA:
            residual.sub_(torch.randn(n, device=DEVICE, dtype=DTYPE, generator=generator) * SIGMA)
        grad.add_(x.t().mv(residual))
        remaining -= n
    return grad / float(batch_size)

@torch.no_grad()
def run_trial(algorithm, eta, rho, seed):
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(int(seed))
    delta = -theta_star.clone()
    previous = delta.clone()
    tail = []
    for step in range(1, STEPS + 1):
        momentum = delta - previous
        evaluation = delta + rho * momentum if algorithm == 'nesterov' else delta
        gradient = exact_gradient(evaluation, BATCH_SIZE, generator)
        new_delta = delta - eta * gradient
        if algorithm != 'sgd':
            new_delta.add_(rho * momentum)
        previous, delta = delta, new_delta
        loss = float((0.5 * torch.sum(lam * delta.square())).item())
        if not math.isfinite(loss) or loss > DIVERGENCE_LOSS:
            return False, math.inf, step
        tail.append(loss)
        if len(tail) > WINDOW:
            tail.pop(0)
    final_average = float(np.mean(tail))
    return final_average < STABLE_THRESHOLD, final_average, STEPS

print(run_trial('sgd', float(ETA_GRID[0]), 0.0, SEED))

## Run and cache the scan

SGD does not depend on $\rho$, so it is evaluated once per $(seed,\eta)$ and broadcast across the horizontal grid. Momentum trials use deterministic random streams keyed by algorithm, seed, $B$, $1-\rho$, and $\eta$ index. The shared cross-point therefore uses exactly the same streams as `batch_size.ipynb`. The cache contains every stability flag and final moving-average loss.

In [ ]:
cache_path = OUTPUT_DIR / 'aligned_momentum_gap_results_seedv2.npz'
shape = (len(ALGORITHMS), N_SEEDS, len(ETA_GRID), len(ONE_MINUS_RHO_GRID))
if cache_path.exists():
    cached = np.load(cache_path, allow_pickle=False)
    cached_config = json.loads(str(cached['config']))
    if cached_config != config:
        raise RuntimeError(f'cache configuration mismatch: move or remove {cache_path}')
    config = cached_config
    ALGORITHMS = tuple(cached['algorithms'].tolist())
    ETA_GRID = cached['eta_grid'].copy()
    ONE_MINUS_RHO_GRID = cached['one_minus_rho_grid'].copy()
    BETA, S, SIGMA = float(config['beta']), float(config['s']), float(config['sigma'])
    BATCH_SIZE, N_SEEDS = int(config['batch_size']), int(config['n_seeds'])
    stable = cached['stable'].copy()
    final_average = cached['final_average'].copy()
    steps_done = cached['steps_done'].copy()
    completed = cached['completed'].copy()
    print(f'resuming from {cache_path}')
else:
    stable = np.zeros(shape, dtype=bool)
    final_average = np.full(shape, np.nan)
    steps_done = np.zeros(shape, dtype=np.int32)
    completed = np.zeros((len(ALGORITHMS), len(ONE_MINUS_RHO_GRID)), dtype=bool)

def save_cache():
    np.savez_compressed(
        cache_path, stable=stable, final_average=final_average, steps_done=steps_done,
        completed=completed, algorithms=np.asarray(ALGORITHMS), eta_grid=ETA_GRID,
        one_minus_rho_grid=ONE_MINUS_RHO_GRID, config=json.dumps(config, sort_keys=True),
    )

t0 = time.time()

for a, algorithm in enumerate(ALGORITHMS):
    rho_indices = [0] if algorithm == 'sgd' else range(len(ONE_MINUS_RHO_GRID))
    print(f'=== {DISPLAY[algorithm]} ===', flush=True)
    for g_idx in rho_indices:
        if completed[a, g_idx]:
            print(f'  skipping completed index {g_idx}', flush=True)
            continue
        g = float(ONE_MINUS_RHO_GRID[g_idx])
        rho = 0.0 if algorithm == 'sgd' else 1.0 - g
        print(f'  1-rho={g:.6g}', flush=True)
        for seed_idx in range(N_SEEDS):
            for eta_idx, eta in enumerate(ETA_GRID):
                trial_seed = aligned_trial_seed(a, seed_idx, BATCH_SIZE, g, eta_idx)
                out = run_trial(algorithm, float(eta), rho, trial_seed)
                stable[a, seed_idx, eta_idx, g_idx] = out[0]
                final_average[a, seed_idx, eta_idx, g_idx] = out[1]
                steps_done[a, seed_idx, eta_idx, g_idx] = out[2]
        completed[a, g_idx] = True
        save_cache()
    if algorithm == 'sgd':
        stable[a] = np.repeat(stable[a, :, :, :1], len(ONE_MINUS_RHO_GRID), axis=2)
        final_average[a] = np.repeat(final_average[a, :, :, :1], len(ONE_MINUS_RHO_GRID), axis=2)
        steps_done[a] = np.repeat(steps_done[a, :, :, :1], len(ONE_MINUS_RHO_GRID), axis=2)
        completed[a, :] = True
        save_cache()

elapsed = time.time() - t0
save_cache()
with (OUTPUT_DIR / 'momentum_gap_config.json').open('w') as f:
    json.dump(config | {'elapsed_seconds': elapsed}, f, indent=2)
print(f'finished in {elapsed:.1f}s; cache saved to {OUTPUT_DIR.resolve()}')

In [ ]:
per_seed_boundary = np.full(
    (len(ALGORITHMS), N_SEEDS, len(ONE_MINUS_RHO_GRID)),
    np.nan,
)
for a in range(len(ALGORITHMS)):
    for seed_idx in range(N_SEEDS):
        for g_idx in range(len(ONE_MINUS_RHO_GRID)):
            indices = np.flatnonzero(stable[a, seed_idx, :, g_idx])
            if indices.size:
                per_seed_boundary[a, seed_idx, g_idx] = ETA_GRID[indices[-1]]

boundary = np.nanmean(per_seed_boundary, axis=1)
boundary_std = np.nanstd(per_seed_boundary, axis=1)


def fit_linear_one_kink(values, scale, min_points_per_side=2, grid_size=4001):
    """Fit eta_crit = min(level, c * scale) with unit log-log slope."""
    finite = np.isfinite(values) & (values > 0) & np.isfinite(scale) & (scale > 0)
    x, y = np.log(scale[finite]), np.log(values[finite])
    order = np.argsort(x)
    x, y = x[order], y[order]
    if x.size < 2 * min_points_per_side:
        raise ValueError('not enough finite points to fit both sides of one kink')

    candidates = np.linspace(
        x[min_points_per_side - 1],
        x[-min_points_per_side],
        grid_size,
    )
    best = None
    for log_breakpoint in candidates:
        hinge = np.minimum(x - log_breakpoint, 0.0)
        log_level = float(np.mean(y - hinge))
        residual = y - (log_level + hinge)
        sse = float(residual @ residual)
        if best is None or sse < best['sse_log']:
            level = float(np.exp(log_level))
            breakpoint = float(np.exp(log_breakpoint))
            best = {
                'coefficient': level / breakpoint,
                'level': level,
                'breakpoint_scale': breakpoint,
                'slope': 1.0,
                'sse_log': sse,
            }
    return best


NORMALIZED_SCALE = {
    'polyak': BATCH_SIZE * ONE_MINUS_RHO_GRID,
    'nesterov': BATCH_SIZE**BETA * ONE_MINUS_RHO_GRID,
}
fits = {
    'sgd': {
        'level': float(np.exp(np.nanmean(np.log(boundary[0])))),
        'slope': 0.0,
    },
    'polyak': fit_linear_one_kink(boundary[1], NORMALIZED_SCALE['polyak']),
    'nesterov': fit_linear_one_kink(
        boundary[2],
        NORMALIZED_SCALE['nesterov'],
    ),
}
fits['polyak']['coefficient_vs_one_minus_rho'] = (
    fits['polyak']['coefficient'] * BATCH_SIZE
)
fits['nesterov']['coefficient_vs_one_minus_rho'] = (
    fits['nesterov']['coefficient'] * BATCH_SIZE**BETA
)

np.savez_compressed(
    OUTPUT_DIR / 'aligned_momentum_gap_summary.npz',
    boundary=boundary,
    boundary_std=boundary_std,
    per_seed_boundary=per_seed_boundary,
    algorithms=np.asarray(ALGORITHMS),
    one_minus_rho_grid=ONE_MINUS_RHO_GRID,
    eta_grid=ETA_GRID,
    fits=json.dumps(fits),
)
cross_gap_index = int(np.flatnonzero(np.isclose(ONE_MINUS_RHO_GRID, 2.0 ** -8))[0])
summary = {
    'config': config,
    'fit_model': 'eta_crit = min(level, coefficient * normalized_scale)',
    'normalized_scale': {
        'Polyak': 'B * (1-rho)',
        'Nesterov': 'B**beta * (1-rho)',
    },
    'fits': fits,
    'boundary': {
        DISPLAY[algorithm]: boundary[index].tolist()
        for index, algorithm in enumerate(ALGORITHMS)
    },
    'cross_point': {
        'batch_size': BATCH_SIZE,
        'one_minus_rho': float(ONE_MINUS_RHO_GRID[cross_gap_index]),
        'boundary': {
            DISPLAY[algorithm]: float(boundary[index, cross_gap_index])
            for index, algorithm in enumerate(ALGORITHMS)
        },
    },
}
with (OUTPUT_DIR / 'aligned_momentum_gap_summary.json').open('w') as f:
    json.dump(summary, f, indent=2)
fits


In [ ]:
colors = {'sgd': 'tab:green', 'polyak': 'tab:blue', 'nesterov': 'tab:orange'}
markers = {'sgd': '^', 'polyak': 'o', 'nesterov': 's'}
plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.labelsize': 16,
    'axes.titlesize': 20,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 10,
})
fig, ax = plt.subplots(figsize=(5.0, 4.0))

for a, algorithm in enumerate(ALGORITHMS):
    values = boundary[a]
    mask = np.isfinite(values) & (values > 0)
    ax.plot(
        ONE_MINUS_RHO_GRID[mask],
        values[mask],
        linestyle='None',
        marker=markers[algorithm],
        markersize=6,
        color=colors[algorithm],
        markerfacecolor=colors[algorithm],
        markeredgecolor=colors[algorithm],
        markeredgewidth=1.1,
        label='_nolegend_',
        zorder=3,
    )

    dense_g = np.geomspace(
        ONE_MINUS_RHO_GRID.min(),
        ONE_MINUS_RHO_GRID.max(),
        500,
    )
    if algorithm == 'sgd':
        prediction = np.full_like(dense_g, fits[algorithm]['level'])
    else:
        power = 1.0 if algorithm == 'polyak' else BETA
        normalized_scale = BATCH_SIZE**power * dense_g
        prediction = np.minimum(
            fits[algorithm]['level'],
            fits[algorithm]['coefficient'] * normalized_scale,
        )
    ax.plot(
        dense_g,
        prediction,
        '--',
        linewidth=1.8,
        color=colors[algorithm],
        label=f'{DISPLAY[algorithm]} Prediction',
        zorder=2,
    )

ax.set_xscale('log', base=2)
ax.set_yscale('log')
x_exponents = np.arange(
    int(np.ceil(np.log2(ONE_MINUS_RHO_GRID.min()))),
    int(np.floor(np.log2(ONE_MINUS_RHO_GRID.max()))) + 1,
)
ax.set_xlim(ONE_MINUS_RHO_GRID.min(), ONE_MINUS_RHO_GRID.max())
ax.set_xticks(2.0**x_exponents)
ax.set_xticklabels([rf'$2^{{{exponent}}}$' for exponent in x_exponents])
ax.set_ylim(ETA_GRID.min(), ETA_GRID.max())
ax.set_xlabel(r'Momentum Damping  $1-\rho$')
ax.set_ylabel(r'Stability Boundary  $\eta$')
ax.set_title('Critical learning rates')
ax.grid(False)
ax.legend(loc='lower right', handlelength=2.8)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'aligned_momentum_gap.png', dpi=180, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'aligned_momentum_gap.pdf', bbox_inches='tight')
plt.show()

for algorithm in ('polyak', 'nesterov'):
    print(
        f"{DISPLAY[algorithm]}: normalized c={fits[algorithm]['coefficient']:.6g}, "
        f"raw coefficient versus (1-rho)="
        f"{fits[algorithm]['coefficient_vs_one_minus_rho']:.6g}, "
        f"level={fits[algorithm]['level']:.6g}"
    )


## Reading the result

Polyak and Nesterov use the same normalized one-kink fit as
`batch_size.ipynb`:

$$
\eta^{\rm crit}=\min\{L,cx\},
$$

with $x=B(1-\rho)$ for Polyak and $x=B^\beta(1-\rho)$ for Nesterov.
Consequently, `coefficient=c` is directly comparable across the two sweeps.
For reference, the summary also reports the raw coefficient against $1-\rho$,
equal to $cB$ or $cB^\beta$. The fitted `breakpoint_scale=L/c` is expressed in
the normalized variable $x$, and `level=L` is the deterministic ceiling.
